In [2]:
import os
import time
import logging
from dotenv import load_dotenv
from typing import Dict, Any
from google import genai
import ollama

load_dotenv()
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

prompt = "What is the currency of USA?"

def run_gemini(prompt: str) -> Dict[str, Any]:
    try:
        client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

        start_time = time.time()
        response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
        elapsed_time = time.time() - start_time

        return {
            "response": response.text.strip(),
            "time_seconds": round(elapsed_time, 2),
            "input_tokens": response.usage_metadata.prompt_token_count,
            "output_tokens": response.usage_metadata.candidates_token_count,
            "total_tokens": response.usage_metadata.total_token_count,
        }
    except Exception as e:
        logging.error(f"Gemini API failed: {e}")
        return {"response": "[Gemini Error]", "time_seconds": 0, "input_tokens": 0, "output_tokens": 0, "total_tokens": 0}


def run_ollama(prompt: str) -> Dict[str, Any]:
    try:
        start_time = time.time()
        response = ollama.chat(
            model="qwen3:4b",
            messages=[{"role": "user", "content": prompt}]
        )
        elapsed_time = time.time() - start_time

        return {
            "response": response["message"]["content"].strip(),
            "time_seconds": round(elapsed_time, 2),
            "input_tokens": response.get("prompt_eval_count", 0),
            "output_tokens": response.get("eval_count", 0),
            "total_tokens": response.get("prompt_eval_count", 0) + response.get("eval_count", 0),
        }
    except Exception as e:
        logging.error(f"Ollama failed: {e}")
        return {"response": "[Ollama Error]", "time_seconds": 0, "input_tokens": 0, "output_tokens": 0, "total_tokens": 0}

def run_ollama_dpsk(prompt: str) -> Dict[str, Any]:
    try:
        start_time = time.time()
        response = ollama.chat(
            model="deepseek-r1",
            messages=[{"role": "user", "content": prompt}]
        )
        elapsed_time = time.time() - start_time

        return {
            "response": response["message"]["content"].strip(),
            "time_seconds": round(elapsed_time, 2),
            "input_tokens": response.get("prompt_eval_count", 0),
            "output_tokens": response.get("eval_count", 0),
            "total_tokens": response.get("prompt_eval_count", 0) + response.get("eval_count", 0),
        }
    except Exception as e:
        logging.error(f"Ollama failed: {e}")
        return {"response": "[Ollama Error]", "time_seconds": 0, "input_tokens": 0, "output_tokens": 0, "total_tokens": 0}


def run_all_models(prompt: str) -> Dict[str, Any]:
    logging.info("Running prompt across Gemini and Ollama...")
    results = {
        "Google Gemini": run_gemini(prompt),
        "Ollama": run_ollama(prompt),
        "Ollama_dpsk": run_ollama_dpsk(prompt)
    }
    return results


if __name__ == "__main__":
    results = run_all_models(prompt)

    print("\n\n>>>>> Comparison of Outputs <<<<<")
    for model, data in results.items():
        print(f"\n--- {model} ---")
        print(f"Response     : {data['response']}")
        print(f"Time Taken   : {data['time_seconds']}s")
        print(f"Input Tokens : {data['input_tokens']}")
        print(f"Output Tokens: {data['output_tokens']}")
        print(f"Total Tokens : {data['total_tokens']}")

2026-03-29 14:25:48,091 - INFO - Running prompt across Gemini and Ollama...
2026-03-29 14:25:48,578 - INFO - AFC is enabled with max remote calls: 10.
2026-03-29 14:25:49,197 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-03-29 14:26:07,787 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-03-29 14:26:47,868 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"




>>>>> Comparison of Outputs <<<<<

--- Google Gemini ---
Response     : The currency of the USA is the **United States Dollar**.
Time Taken   : 0.62s
Input Tokens : 8
Output Tokens: 12
Total Tokens : 38

--- Ollama ---
Response     : The official currency of the **United States of America (USA)** is the **United States dollar** (symbol: **$** or **USD**).  

### Key details:
- **Official Name**: United States dollar  
- **Symbol**: $ (e.g., $100 bill)  
- **ISO Code**: USD  
- **Issued by**: The U.S. Federal Reserve (central bank)  
- **Common Denominations**: Coins (e.g., $0.01, $0.05, $1) and banknotes (e.g., $1, $5, $10, $20, $50, $100).  

### Why this matters:
- The U.S. dollar is the **world's largest reserve currency** and is widely used in international trade, oil pricing, and global financial systems.  
- It is **not** interchangeable with other currencies (e.g., euros, yen, pounds) in U.S. transactions.  

💡 **Quick Tip**: If you see "$" on a bill, receipt, or website, that